# Experiment 1: Llama 3.1-8B with EOS fix
This isolated rerun uses terminal-EOS supervision, EOS-aware generation, the maintained training stack, and a maximum corpus size of 50,000.
Run from `src/eval/`. Results, adapters, and figures use dedicated `llamaeosfix` paths.


# Experiment 1 Evaluation (Prefix Prompts)


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

RESULTS_ROOT = Path("../../results/experiment1-llamaeosfix")

BATCH_SIZE   = 32
FINAL_EPOCH  = 100
SAMPLE_SIZES = [100, 500, 1000, 5000, 10000, 50000]

VARIANT = "prefix_10"
VARIANT_LABEL = "prefix"

# Font size for legends, axis labels and tick labels across all plots
FONT_SIZE = 14

# Where the figure PDFs are written.
FIG_DIR = Path("../../thesis/figures/results/llamaeosfix/experiment1")

In [ ]:
def _load_json(path):
    if not path.exists():
        return None
    with open(path) as f:
        return json.load(f)


def best_epoch(variant, n_samples, batch_size):
    """Epoch with highest Acc@1."""
    base = RESULTS_ROOT / variant / str(n_samples) / str(batch_size)
    if not base.is_dir():
        return None
    best_acc, best_ep = -1.0, None
    for ep_dir in sorted(base.iterdir()):
        if not ep_dir.is_dir():
            continue
        try:
            ep = int(ep_dir.name)
        except ValueError:
            continue
        d = _load_json(ep_dir / "verification_closed.json")
        if d is None:
            continue
        ranks = np.asarray(d["correct_ranks"], dtype=np.int64)
        if ranks.size == 0:
            continue
        acc1 = float((ranks <= 1).mean())
        if acc1 > best_acc:
            best_acc, best_ep = acc1, ep
    return best_ep


def load_correctq_margin_for_epoch(variant, n_samples, batch_size, epoch):
    """Return {'correct_q': mean correct-key q-score,
                'margin':    mean (top1 - top2) score,
                '_epoch':    epoch} for one epoch dir, or None."""
    if epoch is None:
        return None
    path = (RESULTS_ROOT / variant / str(n_samples) / str(batch_size)
            / str(epoch) / "verification_closed.json")
    d = _load_json(path)
    if d is None:
        return None
    correct_scores = np.asarray(d["correct_scores"], dtype=np.float64)
    top_scores     = np.asarray(d["top_scores"],     dtype=np.float64)
    if correct_scores.size == 0 or top_scores.shape[0] == 0:
        return None
    margins = top_scores[:, 0] - top_scores[:, 1]
    return {
        "correct_q": float(correct_scores.mean()),
        "margin":    float(margins.mean()),
        "_epoch":    epoch,
    }


def collect_correctq_margin(variant, batch_size=BATCH_SIZE, sample_sizes=SAMPLE_SIZES):
    """-> {n_samples: {'correct_q', 'margin', '_epoch'}} at the best checkpoint."""
    out = {}
    for n in sample_sizes:
        be = best_epoch(variant, n, batch_size)
        m = load_correctq_margin_for_epoch(variant, n, batch_size, be)
        if m is not None:
            out[n] = m
    return out

In [ ]:

TOPK_LIST = [1, 5, 10, 100]

VARIANT_LABELS = {
    "prefix_10": "prefix",
}


def metrics_from_ranks(ranks, K, topk_list=TOPK_LIST):
    """Return {'acc@k': float, ..., 'mrr': float}. NaN where k > K."""
    out = {}
    for k in topk_list:
        out[f"acc@{k}"] = float((ranks <= k).mean()) if K >= k else float("nan")
    out["mrr"] = float((1.0 / ranks).mean())
    return out


def load_metrics_for_epoch(variant, n_samples, batch_size, epoch):
    if epoch is None:
        return None
    path = (RESULTS_ROOT / variant / str(n_samples) / str(batch_size)
            / str(epoch) / "verification_closed.json")
    d = _load_json(path)
    if d is None:
        return None
    ranks = np.asarray(d["correct_ranks"], dtype=np.int64)
    if ranks.size == 0:
        return None
    return metrics_from_ranks(ranks, int(d["top_k"]))


def load_bm25_metrics_for_epoch(variant, n_samples, batch_size, epoch,
                                source, corpus_type="original"):
    """Load BM25 metrics from `bm25_{source}_{corpus_type}.json` for one epoch dir.
    `source` is "answer" or "prompt".
    """
    if epoch is None:
        return None
    path = (RESULTS_ROOT / variant / str(n_samples) / str(batch_size)
            / str(epoch) / f"bm25_{source}_{corpus_type}.json")
    d = _load_json(path)
    if d is None:
        return None
    ranks = np.asarray(d["correct_ranks"], dtype=np.int64)
    if ranks.size == 0:
        return None
    return metrics_from_ranks(ranks, int(d["top_k"]))


def collect_variant(variant, batch_size=BATCH_SIZE, sample_sizes=SAMPLE_SIZES):
    """-> {'best': {n: metrics}, 'final': {n: metrics}} keyed by epoch type."""
    out = {"best": {}, "final": {}}
    for n in sample_sizes:
        be = best_epoch(variant, n, batch_size)
        for ep, label in ((be, "best"), (FINAL_EPOCH, "final")):
            m = load_metrics_for_epoch(variant, n, batch_size, ep)
            if m is not None:
                m["_epoch"] = ep
                out[label][n] = m
    return out


def collect_bm25(variant, batch_size=BATCH_SIZE, sample_sizes=SAMPLE_SIZES,
                 corpus_type="original"):
    """Collect BM25 metrics at the best watermark epoch for each n_samples.
    Returns {'answer': {n: metrics}, 'prompt': {n: metrics}}.
    """
    out = {"answer": {}, "prompt": {}}
    for n in sample_sizes:
        be = best_epoch(variant, n, batch_size)
        for source in ("answer", "prompt"):
            m = load_bm25_metrics_for_epoch(variant, n, batch_size, be,
                                            source, corpus_type)
            if m is not None:
                m["_epoch"] = be
                out[source][n] = m
    return out


def plot_all_metrics(by_n, title, save_path=None,
                     topk_list=TOPK_LIST,
                     acc_colors=("C0", "C1", "C2", "C3"),
                     mrr_color="C4"):
    """Plot Acc@k for each k in `topk_list` plus MRR.
    """
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    xs = sorted(by_n.keys())
    if xs:
        for k, color in zip(topk_list, acc_colors):
            ys = [by_n[x].get(f"acc@{k}", float("nan")) for x in xs]
            ax.plot(xs, ys,
                    linestyle="-", marker="o", color=color, lw=2.0,
                    label=f"Acc@{k}")
        ax.plot(xs, [by_n[x]["mrr"] for x in xs],
                linestyle="--", marker="s", color=mrr_color, lw=2.0,
                label="MRR")
    ax.set_xscale("log")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(title)
    ax.set_xlabel("N", fontsize=FONT_SIZE)
    ax.set_ylabel("score", fontsize=FONT_SIZE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(alpha=0.3, which="both")
    ax.legend(loc="lower left", fontsize=FONT_SIZE)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()


def plot_acc1_with_bm25(by_n, bm25_by_source, title, save_path=None,
                        acc1_color="C0",
                        bm25_answer_color="C2", bm25_prompt_color="C3"):
    """Plot watermark Acc@1 alongside the two BM25 top-1 baselines.
    """
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    xs = sorted(by_n.keys())
    if xs:
        ax.plot(xs, [by_n[x]["acc@1"] for x in xs],
                linestyle="-", marker="o", color=acc1_color, lw=2.0,
                label="Acc@1")

    ba = bm25_by_source.get("answer", {})
    bp = bm25_by_source.get("prompt", {})
    xs_ba = sorted(ba.keys())
    xs_bp = sorted(bp.keys())
    if xs_ba:
        ax.plot(xs_ba, [ba[x]["acc@1"] for x in xs_ba],
                linestyle="-", marker="o", color=bm25_answer_color, lw=2.0,
                label="BM25PostGen Acc@1")
    if xs_bp:
        ax.plot(xs_bp, [bp[x]["acc@1"] for x in xs_bp],
                linestyle=":", marker="^", color=bm25_prompt_color, lw=2.0,
                label="BM25PostRet Acc@1")

    ax.set_xscale("log")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(title)
    ax.set_xlabel("N", fontsize=FONT_SIZE)
    ax.set_ylabel("score", fontsize=FONT_SIZE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(alpha=0.3, which="both")
    ax.legend(loc="lower left", fontsize=FONT_SIZE)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()

## Acc@k & MRR for WMCite and BM25

In [ ]:
cit_variant = "prefix_10"
cit_data = collect_variant(cit_variant)
cit_bm25 = collect_bm25(cit_variant)

# Acc@k & MRR, best checkpoint
plot_all_metrics(
    cit_data["best"],
    title="Acc@k and MRR",
    save_path=FIG_DIR / "exp1_acck_mrr_prefix.pdf",
)

# Acc@1 vs. the two BM25 baselines, best checkpoint
plot_acc1_with_bm25(
    cit_data["best"],
    cit_bm25,
    title="Acc@1 vs. BM25 baselines",
    save_path=FIG_DIR / "exp1_bm25_baselines_prefix.pdf",
)

## CorrectQ and Margin

In [ ]:
def plot_correctq_margin(by_n, title, save_path=None,
                         correctq_color="C0", margin_color="C4"):
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    xs = sorted(by_n.keys())
    if xs:
        ax.plot(xs, [by_n[x]["correct_q"] for x in xs],
                linestyle="-", marker="o", color=correctq_color, lw=2.0,
                label="CorrectQ")
        ax.plot(xs, [by_n[x]["margin"] for x in xs],
                linestyle="--", marker="s", color=margin_color, lw=2.0,
                label="Margin")
    ax.set_xscale("log")
    ax.set_ylim(bottom=0.0)
    ax.set_title("q-score metrics")
    ax.set_xlabel("N", fontsize=FONT_SIZE)
    ax.set_ylabel("q-score", fontsize=FONT_SIZE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(alpha=0.3, which="both")
    ax.legend(loc="best", fontsize=FONT_SIZE)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
data = collect_correctq_margin(VARIANT)

for n in sorted(data):
    row = data[n]
    print(f"N={n:6d}  epoch={row['_epoch']:>3}  "
          f"correct_q={row['correct_q']:.4f}  margin={row['margin']:.4f}")

plot_correctq_margin(
    data,
    title=f"{VARIANT_LABEL} : best checkpoint (correct-key q-score and top-1/top-2 margin)",
    save_path=FIG_DIR / "exp1_correctq_margin_prefix.pdf",
)

## Semantic and Lexical Metrics

In [ ]:
def load_genmetrics_for_epoch(variant, n_samples, batch_size, epoch):
    """Return {'semsim', 'bertscore', 'rouge2', 'normlcs'} for one epoch dir.
    """
    if epoch is None:
        return None
    base = RESULTS_ROOT / variant / str(n_samples) / str(batch_size) / str(epoch)

    cos = _load_json(base / "cosine_watermarked.json")    or _load_json(base / "cosine_watermarked_summary.json")
    bsc = _load_json(base / "bertscore_watermarked.json") or _load_json(base / "bertscore_watermarked_summary.json")
    big = _load_json(base / "bigrams.json")
    lcs = _load_json(base / "lcs_watermarked.json")       or _load_json(base / "lcs_watermarked_summary.json")

    out = {
        "semsim":    None if cos is None else cos.get("mean"),
        "bertscore": None if bsc is None else bsc.get("mean_f1"),
        "rouge2":    None if big is None else big.get("frac_unique"),
        "normlcs":   None if lcs is None else lcs.get("mean_norm_lcs"),
        "_epoch":    epoch,
    }
    if all(out[k] is None for k in ("semsim", "bertscore", "rouge2", "normlcs")):
        return None
    return out


def collect_genmetrics(variant, batch_size=BATCH_SIZE, sample_sizes=SAMPLE_SIZES):
    """-> {n_samples: {'semsim', 'bertscore', 'rouge2', 'normlcs', '_epoch'}}
    at the best watermark checkpoint."""
    out = {}
    for n in sample_sizes:
        be = best_epoch(variant, n, batch_size)
        m = load_genmetrics_for_epoch(variant, n, batch_size, be)
        if m is not None:
            out[n] = m
    return out

In [ ]:
def plot_genmetrics(by_n, title, save_path=None,
                    sem_color="C1", lex_color="C3"):
    series = [
        ("semsim",    "SemSim",    sem_color, "-",  "o"),
        ("bertscore", "BERTScore", sem_color, ":",  "^"),
        ("rouge2",    "Rouge2",    lex_color, "-",  "o"),
        ("normlcs",   "NormLCS",   lex_color, ":",  "^"),
    ]
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    xs = sorted(by_n.keys())
    if xs:
        for key, label, color, ls, marker in series:
            ys = [by_n[x].get(key, float("nan")) for x in xs]
            ax.plot(xs, ys, linestyle=ls, marker=marker, color=color, lw=2.0,
                    label=label)
    ax.set_xscale("log")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title("semantic and lexical metrics")
    ax.set_xlabel("N", fontsize=FONT_SIZE)
    ax.set_ylabel("score", fontsize=FONT_SIZE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(alpha=0.3, which="both")
    ax.legend(loc="best", fontsize=FONT_SIZE)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()


gen = collect_genmetrics(VARIANT)

for n in sorted(gen):
    r = gen[n]
    print(f"N={n:6d}  epoch={r['_epoch']:>3}  "
          f"semsim={r['semsim']:.4f}  bertscore={r['bertscore']:.4f}  "
          f"rouge2={r['rouge2']:.4f}  normlcs={r['normlcs']:.4f}")

plot_genmetrics(
    gen,
    title=f"{VARIANT_LABEL} : best checkpoint (generation-quality metrics)",
    save_path=FIG_DIR / "exp1_genmetrics_prefix.pdf",
)

## NormRank & NormLen

In [ ]:
def load_rank_len_for_epoch(variant, n_samples, batch_size, epoch):
    """Return {'normrank': mean(rank / n_candidates),
                'normlen':  mean output/ground-truth length,
                '_epoch':   epoch} for one epoch dir, or None.
    """
    if epoch is None:
        return None
    base = RESULTS_ROOT / variant / str(n_samples) / str(batch_size) / str(epoch)

    d = _load_json(base / "verification_closed.json")
    normrank = None
    if d is not None:
        ranks = np.asarray(d["correct_ranks"], dtype=np.float64)
        if ranks.size:
            normrank = float((ranks / d["n_candidates"]).mean())

    lcs = _load_json(base / "lcs_watermarked.json") or _load_json(base / "lcs_watermarked_summary.json")
    normlen = None if lcs is None else lcs.get("mean_norm_len")

    if normrank is None and normlen is None:
        return None
    return {"normrank": normrank, "normlen": normlen, "_epoch": epoch}


def collect_rank_len(variant, batch_size=BATCH_SIZE, sample_sizes=SAMPLE_SIZES):
    """-> {n_samples: {'normrank', 'normlen', '_epoch'}} at the best checkpoint."""
    out = {}
    for n in sample_sizes:
        be = best_epoch(variant, n, batch_size)
        m = load_rank_len_for_epoch(variant, n, batch_size, be)
        if m is not None:
            out[n] = m
    return out

In [ ]:
def plot_rank_len(by_n, title, save_path=None,
                  rank_color="C0", len_color="C3"):
    """Twin-axis plot: x = corpus size (log).
    """
    xs = sorted(by_n.keys())
    fig, ax_r = plt.subplots(figsize=(7.0, 4.5))
    ax_l = ax_r.twinx()

    l_rank = l_len = None
    if xs:
        (l_rank,) = ax_r.plot(
            xs, [by_n[x]["normrank"] for x in xs],
            linestyle="-", marker="o", color=rank_color, lw=2.0, label="NormRank")
        (l_len,) = ax_l.plot(
            xs, [by_n[x]["normlen"] for x in xs],
            linestyle="--", marker="s", color=len_color, lw=2.0, label="NormLen")

    ax_r.set_xscale("log")
    ax_r.set_title("normalized metrics")
    ax_r.set_xlabel("N", fontsize=FONT_SIZE)

    ax_r.set_ylabel("NormRank", color=rank_color, fontsize=FONT_SIZE)
    ax_r.set_ylim(-0.02, 1.02)
    ax_r.tick_params(axis="x", labelsize=FONT_SIZE)
    ax_r.tick_params(axis="y", colors=rank_color, labelsize=FONT_SIZE)

    ax_l.set_ylabel("NormLen", color=len_color, fontsize=FONT_SIZE)
    ax_l.set_ylim(bottom=0.0)
    ax_l.tick_params(axis="y", colors=len_color, labelsize=FONT_SIZE)

    ax_r.grid(alpha=0.3, which="both")
    handles = [h for h in (l_rank, l_len) if h is not None]
    if handles:
        ax_r.legend(handles, [h.get_label() for h in handles],
                    loc="upper left", fontsize=FONT_SIZE)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()


rl = collect_rank_len(VARIANT)

# Quick numeric check
for n in sorted(rl):
    r = rl[n]
    print(f"N={n:6d}  epoch={r['_epoch']:>3}  "
          f"normrank={r['normrank']:.4f}  normlen={r['normlen']:.4f}")

plot_rank_len(
    rl,
    title=f"{VARIANT_LABEL} — best checkpoint (NormRank and NormLen)",
    save_path=FIG_DIR / "exp1_normrank_normlen_prefix.pdf",
)

## Open Keyspace Evaluation
Not run for the EOS-fix arm. The closed-set and auxiliary analyses below remain fully runnable.


In [ ]:
metrics = {}  # optional open-keyspace metrics


## Correlation Scores

In [ ]:
from scipy.stats import pearsonr, spearmanr
CORR_EPOCHS = list(range(5, 101, 5))   # 20 checkpoints

CORR_METRICS = ["correct_q", "margin", "semsim", "bertscore",
                "rouge2", "normlcs", "normlen", "mrr"]
CORR_LABELS = {
    "correct_q": "CorrectQ",
    "margin":    "Margin",
    "semsim":    "SemSim",
    "bertscore": "BERTScore",
    "rouge2":    "Rouge2",
    "normlcs":   "NormLCS",
    "normlen":   "NormLen",
    "mrr":       "MRR",
}


def collect_cell_metrics(variant, n_samples, batch_size, epoch):
    base_m = load_metrics_for_epoch(variant, n_samples, batch_size, epoch)
    if base_m is None:
        return None  # no verification_closed.json -> no Acc@1 to correlate
    cqm = load_correctq_margin_for_epoch(variant, n_samples, batch_size, epoch) or {}
    gen = load_genmetrics_for_epoch(variant, n_samples, batch_size, epoch) or {}
    rl  = load_rank_len_for_epoch(variant, n_samples, batch_size, epoch) or {}
    return {
        "acc@1":     base_m["acc@1"],
        "mrr":       base_m["mrr"],
        "correct_q": cqm.get("correct_q"),
        "margin":    cqm.get("margin"),
        "semsim":    gen.get("semsim"),
        "bertscore": gen.get("bertscore"),
        "rouge2":    gen.get("rouge2"),
        "normlcs":   gen.get("normlcs"),
        "normlen":   rl.get("normlen"),
    }


rows = []
for n in SAMPLE_SIZES:
    for ep in CORR_EPOCHS:
        c = collect_cell_metrics(VARIANT, n, BATCH_SIZE, ep)
        if c is not None:
            rows.append(c)

acc1 = np.array([r["acc@1"] for r in rows], dtype=np.float64)
print(f"Collected {len(rows)} (size, epoch) cells "
      f"({len(SAMPLE_SIZES)} sizes x {len(CORR_EPOCHS)} checkpoints, batch={BATCH_SIZE}).")

header = f"{'metric':>10s} {'n':>4s} {'Pearson r':>10s} {'Spearman r':>11s}"
print(header)
print("-" * len(header))
corr_results = {}
for key in CORR_METRICS:
    other = np.array([r[key] if r[key] is not None else np.nan for r in rows],
                     dtype=np.float64)
    mask = np.isfinite(acc1) & np.isfinite(other)
    n_pairs = int(mask.sum())
    if n_pairs < 2:
        print(f"{CORR_LABELS[key]:>10s} {n_pairs:>4d} {'—':>10s} {'—':>11s}")
        continue
    pr = pearsonr(acc1[mask], other[mask])[0]
    sr = spearmanr(acc1[mask], other[mask])[0]
    corr_results[key] = {"n": n_pairs, "pearson": pr, "spearman": sr}
    print(f"{CORR_LABELS[key]:>10s} {n_pairs:>4d} {pr:>10.4f} {sr:>11.4f}")


## Training Trajectories

In [ ]:
from matplotlib.lines import Line2D

ADAPTERS_ROOT = Path("../../lora_adapters/llamaeosfix/abstracts_only")
TRAJ_EPOCHS   = list(range(5, 101, 5))

SIZE_COLORS = {
    100:   "#e41a1c",  # red
    500:   "#ff7f00",  # orange
    1000:  "#4daf4a",  # green
    5000:  "#377eb8",  # blue
    10000: "#984ea3",  # purple
    50000: "#a65628",  # brown
}


def collect_trajectory(variant, n_samples, batch_size, epochs=TRAJ_EPOCHS):
    """Per-epoch Acc@1 and NormRank for one corpus size, across every epoch dir.
    """
    eps, acc1, normrank = [], [], []
    for ep in epochs:
        d = _load_json(RESULTS_ROOT / variant / str(n_samples) / str(batch_size)
                       / str(ep) / "verification_closed.json")
        if d is None:
            continue
        ranks = np.asarray(d["correct_ranks"], dtype=np.float64)
        if ranks.size == 0:
            continue
        eps.append(ep)
        acc1.append(float((ranks <= 1).mean()))
        normrank.append(float((ranks / d["n_candidates"]).mean()))
    return {
        "epoch":    np.asarray(eps, dtype=np.int64),
        "acc1":     np.asarray(acc1, dtype=np.float64),
        "normrank": np.asarray(normrank, dtype=np.float64),
    }


def load_loss_trajectory(n_samples, batch_size, final_epoch=FINAL_EPOCH):
    """Per-epoch evaluated train loss and held-out loss from the final
    checkpoint's `trainer_state.json`"""
    state = _load_json(ADAPTERS_ROOT / str(n_samples) / str(batch_size)
                       / str(final_epoch) / "lora_adapter" / "trainer_state.json")
    if state is None:
        return None
    log = state["log_history"]
    te = np.array([e["epoch"]           for e in log if "eval_train_loss" in e])
    tl = np.array([e["eval_train_loss"] for e in log if "eval_train_loss" in e])
    he = np.array([e["epoch"]             for e in log if "eval_heldout_loss" in e])
    hl = np.array([e["eval_heldout_loss"] for e in log if "eval_heldout_loss" in e])
    if hl.size == 0:
        he = np.array([e["epoch"]     for e in log if "eval_loss" in e])
        hl = np.array([e["eval_loss"] for e in log if "eval_loss" in e])
    return {"train_epoch": te, "train_loss": tl, "held_epoch": he, "held_loss": hl}

In [ ]:
def plot_acc1_trajectory(variant, save_path=None, epochs=TRAJ_EPOCHS):
    """Acc@1 vs. epoch"""
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    for n in SAMPLE_SIZES:
        t = collect_trajectory(variant, n, BATCH_SIZE, epochs)
        if t["epoch"].size == 0:
            continue
        ax.plot(t["epoch"], t["acc1"], color=SIZE_COLORS[n], marker="o",
                ms=4, lw=2.0, label=f"N={n:,}")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title("Acc@1 over training")
    ax.set_xlabel("epoch", fontsize=FONT_SIZE)
    ax.set_ylabel("Acc@1", fontsize=FONT_SIZE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(alpha=0.3, which="both")
    ax.legend(loc="best", ncol=2, fontsize=FONT_SIZE)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()


plot_acc1_trajectory(VARIANT, save_path=FIG_DIR / "exp1_acc1_trajectory_prefix.pdf")

In [ ]:
def plot_normrank_trajectory(variant, save_path=None, epochs=TRAJ_EPOCHS):
    """NormRank (mean rank / corpus size) vs. epoch.
    """
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    for n in SAMPLE_SIZES:
        t = collect_trajectory(variant, n, BATCH_SIZE, epochs)
        if t["epoch"].size == 0:
            continue
        ax.plot(t["epoch"], t["normrank"], color=SIZE_COLORS[n], marker="o",
                ms=4, lw=2.0, label=f"N={n:,}")
    ax.set_ylim(bottom=0.0)
    ax.set_title("NormRank over training")
    ax.set_xlabel("epoch", fontsize=FONT_SIZE)
    ax.set_ylabel("NormRank", fontsize=FONT_SIZE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(alpha=0.3, which="both")
    #ax.legend( ncol=2, fontsize=FONT_SIZE, loc = "center right")
    ax.legend(ncol=2, fontsize=FONT_SIZE, loc="center right", bbox_to_anchor=(0.98, 0.68))
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()


plot_normrank_trajectory(VARIANT, save_path=FIG_DIR / "exp1_normrank_trajectory_prefix.pdf")

In [ ]:
def plot_loss_trajectory(save_path=None):
    """Train and held-out loss vs. epoch (log y):
    """
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    for n in SAMPLE_SIZES:
        ld = load_loss_trajectory(n, BATCH_SIZE)
        if ld is None:
            continue
        c = SIZE_COLORS[n]
        if ld["train_loss"].size:
            ax.plot(ld["train_epoch"], ld["train_loss"], color=c,
                    linestyle="-", lw=3, alpha=0.75)
        if ld["held_loss"].size:
            ax.plot(ld["held_epoch"], ld["held_loss"], color=c,
                    linestyle=":", lw=3, alpha=0.75)
    ax.set_yscale("log")
    ax.set_title("Loss trajectories during training")
    ax.set_xlabel("epoch", fontsize=FONT_SIZE)
    ax.set_ylabel("loss (log)", fontsize=FONT_SIZE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(alpha=0.3, which="both")

    # Two-part legend: color -> corpus size, linestyle -> train / held-out.
    color_handles = [Line2D([0], [0], color=SIZE_COLORS[n], lw=2.0,
                            label=f"N={n:,}") for n in SAMPLE_SIZES]
    style_handles = [
        Line2D([0], [0], color="black", lw=4.0, linestyle="-", label="train"),
        Line2D([0], [0], color="black", lw=4.0, linestyle=":", label="held-out"),
    ]
    leg = ax.legend(handles=color_handles, loc="center right", ncol=2,
                    fontsize=FONT_SIZE)
    ax.add_artist(leg)
    ax.legend(handles=style_handles, loc="lower left", fontsize=FONT_SIZE)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()


plot_loss_trajectory(save_path=FIG_DIR / "exp1_loss_trajectory_prefix.pdf")

## Summary Table

In [ ]:
TABLE_ROWS = [
    ("Acc@1",             cit_data["best"],   "acc@1"),
    ("Acc@5",             cit_data["best"],   "acc@5"),
    ("Acc@10",            cit_data["best"],   "acc@10"),
    ("Acc@100",           cit_data["best"],   "acc@100"),
    ("MRR",               cit_data["best"],   "mrr"),
    ("BM25PostGen Acc@1", cit_bm25["answer"], "acc@1"),
    ("BM25PostRet Acc@1", cit_bm25["prompt"], "acc@1"),
    ("CorrectQ",          data,               "correct_q"),
    ("Margin",            data,               "margin"),
    ("SemSim",            gen,                "semsim"),
    ("BERTScore",         gen,                "bertscore"),
    ("Rouge2",            gen,                "rouge2"),
    ("NormLCS",           gen,                "normlcs"),
    ("NormRank",          rl,                 "normrank"),
    ("NormLen",           rl,                 "normlen"),
    ("AUROC",             metrics,            "auroc"),
    ("TPR@FPR=0.01",      metrics,            "tpr@0.01"),
    ("TPR@FPR=0.05",      metrics,            "tpr@0.05"),
]


def _fmt(v):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "--"
    return f"{v:.3f}"


def _cell(src, n, key):
    row = src.get(n)
    return None if row is None else row.get(key)


lines = [
    r"\begin{table}[t]",
    r"  \centering",
    r"  \caption{Experiment~1 (prefix) metrics at the best watermark checkpoint "
    r"(max closed-set Acc@1), per corpus size $N$.}",
    r"  \label{tab:exp1_metrics}",
    r"  \begin{tabular}{l" + "r" * len(SAMPLE_SIZES) + "}",
    r"    \toprule",
    "    Metric & " + " & ".join(f"$N={n:,}$" for n in SAMPLE_SIZES) + r" \\",
    r"    \midrule",
]
for label, src, key in TABLE_ROWS:
    cells = " & ".join(_fmt(_cell(src, n, key)) for n in SAMPLE_SIZES)
    lines.append(f"    {label} & {cells} " + r"\\")
lines += [
    r"    \bottomrule",
    r"  \end{tabular}",
    r"\end{table}",
]

latex_table = "\n".join(lines)
print(latex_table)